[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/derrickkimutai1/tb-screening-system/blob/main/ml/notebooks/01_sprint1_data_preparation.ipynb)


# Sprint 1: data preparation and environment

Establishes the foundation the model work depends on, and proves it rather than asserting
it. Everything here is reproducible: the split is seeded, and re-running produces the same
manifest.

**What this sprint had to settle**

1. **Which images, and are they usable.** Montgomery and Shenzhen, both public and
   de-identified, from the U.S. National Library of Medicine. Every image is decoded,
   measured and fingerprinted.
2. **Are any duplicated.** A duplicate crossing a split boundary would put a training image
   into the test set and inflate every figure that follows.
3. **How the data is divided.** 70/15/15, stratified on source dataset and class together,
   seeded so it reproduces exactly.
4. **How an image becomes model input.** One module, `ml/preprocessing.py`, imported by both
   the training scripts and the web application, so the two cannot diverge.

**The finding that shaped the preprocessing**

The datasets are not uniformly encoded. Montgomery is 8-bit greyscale; Shenzhen is 635
palette-mode images and 27 RGB. Loaded without conversion these produce arrays of different
shapes, which would fail outright or silently pass malformed input for about 4% of the data.
Dimensions also vary from 1130x948 to 4892x4892.

This notebook orchestrates and verifies. The logic lives in the repository.

## 0. Before you run

- A GPU is **not** needed for this notebook; it only inspects and splits. Leave the runtime
  on CPU.
- All Sprint 1 code must be pushed to GitHub. The repository check below fails loudly
  otherwise.
- `datasets.zip` must be in Drive at `MyDrive/tb-screening-system/`, written by
  `scripts/backup_to_drive.py` on the laptop.
- Nothing is installed: Colab already provides NumPy, Pillow, scikit-learn and matplotlib.
  Section 1 records the versions actually used.

## 1. Repository and environment

In [ ]:
%%bash
cd /content
if [ -d tb-screening-system ]; then
  cd tb-screening-system && git fetch -q origin && git checkout -q main && git pull -q origin main
else
  git clone -q https://github.com/derrickkimutai1/tb-screening-system.git
fi

In [ ]:
%cd /content/tb-screening-system

from pathlib import Path

required = [
    "ml/preprocessing.py",
    "ml/provenance.py",
    "ml/scripts/download_shenzhen.py",
    "ml/scripts/prepare_data.py",
    "scripts/backup_to_drive.py",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, (
    f"Not in the cloned repository: {missing}. Commit and push from your machine, "
    "then rerun from the top."
)
print("Repository check passed.")

In [ ]:
import subprocess
import sys

import numpy as np
import PIL
import sklearn

sys.path.insert(0, "ml")
import provenance  # noqa: E402

COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()

print("commit       ", COMMIT[:12])
print("python       ", sys.version.split()[0])
print("numpy        ", np.__version__)
print("pillow       ", PIL.__version__)
print("scikit-learn ", sklearn.__version__)

## 2. Configuration

In [ ]:
# --- Run identity -----------------------------------------------------------
RUN_TAG = "sprint1-data"

# --- Split ------------------------------------------------------------------
# These mirror ml/scripts/prepare_data.py. Changing them here does not change the
# split; edit the script, push, and rerun, so the manifest always matches the code.
SEED = 42
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15

# --- What to do this session ------------------------------------------------
REBUILD_SPLIT = False  # True regenerates splits.csv; False verifies the committed one

# --- Where results go -------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/tb-screening-system"

## 3. Google Drive

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)
print("drive root :", DRIVE_ROOT)
print("run folder :", RUN_DIR)
print("contents   :", sorted(os.listdir(DRIVE_ROOT)))

## 4. Load the datasets

Copied from Drive as one archive and unpacked locally. Reading 800 files individually
through the Drive mount is far slower than copying one archive.

In [ ]:
import shutil
import zipfile

archive = f"{DRIVE_ROOT}/datasets.zip"
assert os.path.exists(archive), (
    "datasets.zip is not in Drive. Run scripts/backup_to_drive.py on the laptop first."
)

if not os.path.isdir("data/raw/shenzhen"):
    shutil.copy(archive, "/content/datasets.zip")
    with zipfile.ZipFile("/content/datasets.zip") as bundle:
        bundle.extractall(".")

montgomery = sorted(Path("data/raw/montgomery/MontgomerySet/CXR_png").glob("MCUCXR_*.png"))
shenzhen = sorted(Path("data/raw/shenzhen/CXR_png").glob("CHNCXR_*.png"))
masks = sorted(Path("data/raw/montgomery/MontgomerySet/ManualMask/leftMask").glob("*.png"))

print(f"montgomery  {len(montgomery)} radiographs")
print(f"shenzhen    {len(shenzhen)} radiographs")
print(f"lung masks  {len(masks)} left (and the same number right)")

## 5. Inspect the images

Every image is opened and checked. The label comes from the filename: both datasets end
each name in `_0` for normal or `_1` for TB.

In [ ]:
from collections import Counter

from PIL import Image

Image.MAX_IMAGE_PIXELS = None

records = []
for source, paths in (("montgomery", montgomery), ("shenzhen", shenzhen)):
    for path in paths:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            records.append(
                {
                    "source": source,
                    "label": int(path.stem[-1]),
                    "mode": image.mode,
                    "width": image.size[0],
                    "height": image.size[1],
                }
            )

print(f"images decoded without error : {len(records)}")
print(f"colour modes present         : {dict(Counter(r['mode'] for r in records))}")
print(f"width  {min(r['width'] for r in records)} to {max(r['width'] for r in records)} pixels")
print(
    f"height {min(r['height'] for r in records)} to {max(r['height'] for r in records)} pixels"
)
print()
for source in ("montgomery", "shenzhen"):
    counts = Counter(r["label"] for r in records if r["source"] == source)
    total = counts[1] + counts[0]
    print(
        f"  {source:<12}{counts[1]:>4} TB  {counts[0]:>4} normal   {counts[1] / total:.1%} TB"
    )
counts = Counter(r["label"] for r in records)
total = counts[1] + counts[0]
print(f"  {'all':<12}{counts[1]:>4} TB  {counts[0]:>4} normal   {counts[1] / total:.1%} TB")

## 6. Check for duplicates

By content hash, within each dataset and across the two. This is the check that protects
every later figure: an image appearing in both training and test would mean the model is
graded on something it has already seen.

In [ ]:
import hashlib

digests = {}
for source, paths in (("montgomery", montgomery), ("shenzhen", shenzhen)):
    for path in paths:
        digest = hashlib.md5(path.read_bytes()).hexdigest()
        digests.setdefault(digest, []).append(f"{source}/{path.name}")

duplicates = {d: names for d, names in digests.items() if len(names) > 1}
print(f"distinct images : {len(digests)} of {len(records)} files")
print(f"duplicate groups: {len(duplicates)}")
for names in list(duplicates.values())[:5]:
    print("   ", ", ".join(names))

## 7. The split

70/15/15, stratified on **source dataset and class together**, so each split holds the same
proportion of Montgomery to Shenzhen and of TB to normal. With a test set of only 120
images an unstratified split can drift badly on both counts.

Both datasets hold one image per patient, so an image-level split is also a patient-level
split.

In [ ]:
if REBUILD_SPLIT:
    !python ml/scripts/prepare_data.py
else:
    print("Verifying the committed split. Set REBUILD_SPLIT = True to regenerate it.")

import csv

with open("ml/artifacts/splits.csv", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

print(f"\n{'split':<13}{'images':>8}{'TB':>6}{'normal':>8}{'TB share':>11}")
for split in ("train", "validation", "test"):
    subset = [r for r in rows if r["split"] == split]
    tb = sum(1 for r in subset if r["label"] == "1")
    print(f"  {split:<11}{len(subset):>8}{tb:>6}{len(subset) - tb:>8}{tb / len(subset):>10.1%}")
print(f"  {'total':<11}{len(rows):>8}")
print()
print("dataset fingerprint:", provenance.dataset_fingerprint())

In [ ]:
# No image belongs to more than one split, and the three splits account for every image.
from collections import defaultdict

seen = defaultdict(set)
for row in rows:
    seen[row["filename"]].add(row["split"])

overlapping = {name: splits for name, splits in seen.items() if len(splits) > 1}
print(f"images in more than one split : {len(overlapping)}")
print(f"images accounted for          : {len(seen)} of {len(records)}")
assert not overlapping, "An image appears in more than one split."
assert len(seen) == len(records), "The manifest does not cover every image."
print("\nSplit integrity checks passed.")

## 8. Preprocessing

One module prepares images for both training and the web application. Any source encoding
becomes single-channel intensity, is resized to 224x224, repeated across three channels for
MobileNetV2, and scaled to the range -1 to 1 that the ImageNet weights were trained with.

Converting palette images straight to RGB would pass their palette through instead of their
intensity, which is why intensity comes first.

In [ ]:
import json

import numpy as np
import preprocessing

print(json.dumps(preprocessing.describe(), indent=2))

for split in ("train", "validation", "test"):
    subset = [r for r in rows if r["split"] == split][:8]
    batch = preprocessing.prepare_batch(r["filepath"] for r in subset)
    labels = [int(r["label"]) for r in subset]
    print(
        f"  {split:<11}{str(batch.shape):<22}{batch.dtype}   "
        f"range [{batch.min():.2f}, {batch.max():.2f}]"
    )

In [ ]:
# The three source encodings must produce identical output, or roughly 4% of the data
# would be prepared differently from the rest.
import io

grey = Image.new("L", (300, 300), color=128)
outputs = []
for mode in ("L", "P", "RGB"):
    buffer = io.BytesIO()
    grey.convert(mode).save(buffer, format="PNG")
    buffer.seek(0)
    outputs.append(preprocessing.prepare(buffer))

identical = all(np.array_equal(outputs[0], other) for other in outputs[1:])
print("greyscale, palette and RGB give identical output:", identical)

# And the scaling must match the function MobileNetV2's weights were trained with.
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input  # noqa: E402

sample = preprocessing.prepare(montgomery[0])
with Image.open(montgomery[0]) as image:
    resized = image.convert("L").resize((224, 224), Image.Resampling.BILINEAR)
raw = np.repeat(np.asarray(resized, dtype=np.float32)[..., np.newaxis], 3, axis=-1)

print(
    "scaling matches keras preprocess_input:",
    np.array_equal(sample, preprocess_input(raw)),
)

## 9. Look at the images

A visual check that labels are attached to the right images and that preparation has not
damaged them.

In [ ]:
import random

import matplotlib.pyplot as plt

random.seed(SEED)
sample = random.sample([r for r in rows if r["split"] == "train"], 8)

figure, axes = plt.subplots(2, 4, figsize=(13, 7))
for axis, row in zip(axes.ravel(), sample, strict=True):
    prepared = preprocessing.prepare(row["filepath"])
    axis.imshow((prepared + 1) / 2)
    axis.set_title(
        f"{'TB' if row['label'] == '1' else 'Normal'}  ({row['source']})", fontsize=10
    )
    axis.axis("off")
figure.suptitle("Prepared training images, 224x224, scaled to -1 to 1", fontsize=12)
plt.tight_layout()
plt.show()

## 10. Save the run summary to Drive

In [ ]:
summary = {
    "run_tag": RUN_TAG,
    "git_commit": COMMIT,
    "provenance": provenance.describe(),
    "ran_on": "colab",
    "versions": {
        "numpy": np.__version__,
        "pillow": PIL.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "datasets": {
        "montgomery": len(montgomery),
        "shenzhen": len(shenzhen),
        "lung_masks": len(masks),
        "total": len(records),
        "tb": sum(1 for r in records if r["label"] == 1),
        "colour_modes": dict(Counter(r["mode"] for r in records)),
    },
    "duplicates": len(duplicates),
    "split": {
        "seed": SEED,
        "test_fraction": TEST_FRACTION,
        "validation_fraction": VALIDATION_FRACTION,
        "counts": {
            split: sum(1 for r in rows if r["split"] == split)
            for split in ("train", "validation", "test")
        },
    },
    "preprocessing": preprocessing.describe(),
    "checks": {
        "all_images_decoded": True,
        "no_duplicates": len(duplicates) == 0,
        "no_split_overlap": len(overlapping) == 0,
        "encodings_identical": bool(identical),
    },
}

with open(f"{RUN_DIR}/run_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

for artefact in ("ml/artifacts/splits.csv", "ml/artifacts/data_summary.md"):
    if os.path.exists(artefact):
        shutil.copy2(artefact, f"{RUN_DIR}/{os.path.basename(artefact)}")

print("saved to", RUN_DIR)
for name in sorted(os.listdir(RUN_DIR)):
    print("   ", name)

## Sprint 1 exit checklist

- [ ] Repository check passed, so the run used pushed code and the commit is recorded.
- [ ] All 800 images decode; none are corrupt.
- [ ] **Zero duplicates** within or across the two datasets.
- [ ] Split is 70/15/15, stratified on source and class, and reproduces from seed 42.
- [ ] **No image appears in more than one split**, and every image is accounted for.
- [ ] Greyscale, palette and RGB inputs produce byte-identical prepared output.
- [ ] Scaling verified identical to `mobilenet_v2.preprocess_input`.
- [ ] Prepared images inspected visually with their labels.
- [ ] `run_summary.json`, `splits.csv` and `data_summary.md` saved to the run folder in Drive.
- [ ] Notebook saved back to GitHub, so the recorded code matches the run.

**If duplicates appear:** do not proceed to training. A duplicate crossing a split boundary
puts a training image into the test set, and every metric afterwards is overstated by an
unknown amount.

**If the encodings do not produce identical output:** roughly 4% of the Shenzhen images are
RGB rather than palette, so a mismatch here means that fraction of the data is prepared
differently from the rest, and the model would be learning from two populations.